In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import math

# DMRG done by my self


## first do exact diagonalization   

In [34]:
N=4#number of spins in chain
sigmaz=np.array([[1,0],[0, -1]])
sigmax=np.array([[0,1],[1, 0]])
id=np.eye(2)
def operator(single_site_operator,index):
    result=1
    for i in range(N):
        if i == index:
            result=np.kron(result,single_site_operator)
        else:
            result=np.kron(result,id)
    return result



$$ H=-J\sum_i^{N-1}\sigma_z^i \sigma_z^{i+1}- h\sum_i^{N}\sigma_x^{i}$$

In [35]:
J=5
h=-1
def ising_hamiltonian():
    H=0
    for i in range(N-1):
        H+=-J*operator(sigmaz,i)@operator(sigmaz,i+1)
    for i in range(N):
        H+=-h*operator(sigmax,i)
    return H


In [36]:
eigenvalues, eigenvector=np.linalg.eigh(ising_hamiltonian())
print(eigenvalues)
def left_right_env2(mps,mpo,center_index):
    L=np.ones((1,1,1))
    R=np.ones((1,1,1))
    #left env loop
    for i in range(center_index):
        L=np.einsum(
            "abc,asd,bste,ctf->def",
            L,
            mps[i].conj(),
            mpo[i],
            mps[i]
        )
    for i in range(len(mps)-1, center_index, -1):
        n=len(mps)
        R = np.einsum(
            "asd,bste,ctf,def->abc",
            mps[i].conj(),
            mpo[i],
            mps[i],
            R
        )
    return L,R
        
def action_H_eff(mps,mpo,center_index):
    L,R=left_right_env2(mps,mpo,center_index)
    return np.einsum(
        "abc,csd,btse,ged->atg",
        L,mps[center_index],mpo[center_index],R
    )

def time_evolution(L,R,mps_tensor,mpo_tensor,deltat=0.01):
    H_eff = np.einsum(
    "abc,btse,ged->atgcsd",
    L, mpo_tensor, R
    )
    A=mps_tensor.reshape(-1)
    H_eff_matrix=H_eff.reshape(len(A),len(A))
    A=sp.linalg.expm(-1j*H_eff_matrix*deltat)@A
    return A.reshape(mps_tensor.shape[0],mps_tensor.shape[1],mps_tensor.shape[2])

    

[-15.30867402 -15.29331387  -6.53750971  -6.52214956  -5.01536015
  -5.          -3.77116431  -3.75580416   3.75580416   3.77116431
   5.           5.01536015   6.52214956   6.53750971  15.29331387
  15.30867402]


creating MPS


In [37]:
def random_mps(number_of_sites=N,bond_dim=4):
    mps=[]
    for i in range(number_of_sites):
        if i==0:
            tensor=np.random.randn(1,2,bond_dim)
        elif i==number_of_sites-1:
            tensor=np.random.randn(bond_dim,2,1)
        else:
            tensor=np.random.randn(bond_dim,2,bond_dim)
        mps.append(tensor)
        return mps
def wavefunction_to_mps(psi=np.random.randn(2**N),number_of_sites=N,max_bond_dim=4):
    
    #psi=np.random.randn(2**number_of_sites)
    psi/=np.linalg.norm(psi)
    psi=psi.reshape([2]*number_of_sites) #rehape([2]*N)
    mps=[]
    tensor=psi
    previous_bond_dim=1
    for i in range(number_of_sites-1):
        if i==0:
            M=tensor.reshape(2,-1)
            U,S,Vh=np.linalg.svd(M,full_matrices=False)
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            tensor=np.diag(S)@Vh
            mps.append(U.reshape(previous_bond_dim,2,bond_dim))
            previous_bond_dim=bond_dim
        elif i==number_of_sites-2:
            M=tensor.reshape(previous_bond_dim*2,-1)
            U,S,Vh=np.linalg.svd(M,full_matrices=False)
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            tensor=np.diag(S)@Vh
            mps.append(U.reshape(previous_bond_dim,2,bond_dim))
            mps.append(tensor.reshape(bond_dim,2,1))
            previous_bond_dim=bond_dim
        else:
            M=tensor.reshape(bond_dim*2,-1)
            U,S,Vh=np.linalg.svd(M,full_matrices=False)
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            tensor=np.diag(S)@Vh
            mps.append(U.reshape(previous_bond_dim,2,bond_dim))
            previous_bond_dim=bond_dim
            

    return mps

def mps_to_state(mps,number_of_sites=N):
    Tensor=mps[0]
    for A in mps[1:]:
        Tensor=np.tensordot(Tensor,A,axes=([-1],[0]))
    #Tensor=np.squeeze(Tensor)
    return Tensor.reshape(-1)







In [38]:
psi=np.random.randn(2**N)
psi/=np.linalg.norm(psi)
Mps=wavefunction_to_mps(psi=psi,max_bond_dim=3)
print(psi)
print(Mps)
print("state reconstructed")
print(mps_to_state(Mps))

[-0.11699641 -0.31275014 -0.14799409 -0.11551072  0.09365988 -0.19559497
 -0.48560537  0.39426394  0.20062822  0.40933873  0.20258053  0.04773303
  0.18422991 -0.03646012 -0.28534731  0.21713533]
[array([[[-0.95130159, -0.3082617 ],
        [-0.3082617 ,  0.95130159]]]), array([[[ 0.12674555, -0.31846031, -0.58960746],
        [ 0.86268467,  0.43341057,  0.18234816]],

       [[ 0.4668953 , -0.79913411,  0.07049797],
        [-0.1473754 , -0.26855724,  0.78367227]]]), array([[[-0.42705597, -0.39743517],
        [ 0.82591398, -0.41095417]],

       [[ 0.10780027,  0.76752315],
        [ 0.34217648,  0.25915702]],

       [[ 0.07083783, -0.08354117],
        [ 0.04201973,  0.09964279]]]), array([[[ 0.51279732],
        [-0.60989871]],

       [[-0.46241481],
        [-0.3887942 ]]])]
state reconstructed
[-0.11486002 -0.31557267 -0.14495372 -0.11371128  0.09492372 -0.19726471
 -0.48380676  0.39532844  0.20275848  0.40652429  0.20561218  0.04952731
  0.18255087 -0.03424182 -0.28773681  0.2

In [39]:
len(Mps)

4

In [40]:
a=Mps[3].shape
print(a[-1])

1


In [41]:
for i in range(4):
    print(i)

0
1
2
3


In [42]:
def tdvp_canonical_form(mps,center_index):
    number_of_tensors=len(mps)
    number_of_tensors_before_center=center_index #indexing starts at zero, hence no need to add or sub 1
    number_of_tensors_after_center=number_of_tensors-(center_index+1)
    for i in range(number_of_tensors_after_center):
    
        A=mps[number_of_tensors-1-i]
        M=A.reshape(A.shape[0],A.shape[1]*A.shape[2])
        Q_r,R=np.linalg.qr(M.conj().T)
        L=R.conj().T
        Q=Q_r.conj().T
        mps[number_of_tensors-i-1]=Q.reshape(A.shape[0],A.shape[1],A.shape[2])
        mps[number_of_tensors-i-2]=np.tensordot(mps[number_of_tensors-i-2],L,axes=(2,0))
    for i in range(center_index):
        
        A=mps[i]
        N=A.reshape(A.shape[0]*A.shape[1],A.shape[2])
        Q,R=np.linalg.qr(N)
        mps[i]=Q.reshape(A.shape[0],A.shape[1],A.shape[2])
        mps[i+1]=np.tensordot(R, mps[i+1], axes=(1, 0))

    

In [43]:
tdvp_canonical_form(Mps,1)

In [44]:
H=ising_hamiltonian()
print(H.shape)
H_tensor=H.reshape(2,2,2,2, 2,2,2,2)
print(H_tensor.shape)

(16, 16)
(2, 2, 2, 2, 2, 2, 2, 2)


In [45]:
def hamiltonian_to_mpo(H,number_of_sites=4,max_bond_dim=4):
    H=H.reshape([2]*number_of_sites+[2]*number_of_sites)
    perm=[]
    for i in range(number_of_sites):
        perm.append(i)
        perm.append(number_of_sites+i)
    Tensor=H.transpose(perm)
    Tensor=Tensor.reshape([4]*number_of_sites)
    mpo=[]
    previous_bond_dim=1
    for i in range(number_of_sites-1):
        if i==0:
            U,S,Vh=np.linalg.svd(Tensor.reshape(4,-1))
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            Tensor=np.diag(S)@Vh
            mpo.append(U.reshape(previous_bond_dim,2,2,bond_dim))
            previous_bond_dim=bond_dim
        elif i==number_of_sites-2:
            U,S,Vh=np.linalg.svd(Tensor.reshape(previous_bond_dim*4,-1))
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            Tensor=np.diag(S)@Vh
            mpo.append(U.reshape(previous_bond_dim,2,2,bond_dim))
            mpo.append(Tensor.reshape(bond_dim,2,2,1))
        else:
            U,S,Vh=np.linalg.svd(Tensor.reshape(previous_bond_dim*4,-1))
            bond_dim=min(max_bond_dim,len(S))
            U=U[:,:bond_dim]
            S=S[:bond_dim]
            Vh=Vh[:bond_dim,:]
            Tensor=np.diag(S)@Vh
            mpo.append(U.reshape(previous_bond_dim,2,2,bond_dim))
            previous_bond_dim=bond_dim
    return mpo
def mpo_to_operator(mpo,number_of_sites=N):
    Tensor=mpo[0][0]
    for A in mpo[1:]:
        Tensor=np.tensordot(Tensor,A,axes=([-1],[0]))
    perm=[]
    Tensor=Tensor[...,0]
    for i in range(number_of_sites):
        perm.append(i)
        perm.append(number_of_sites+i)
    inverse_perm=np.argsort(perm)
    Tensor=Tensor.transpose(inverse_perm)
    return Tensor.reshape(2**number_of_sites,2**number_of_sites) 
    




In [46]:
#N=8
H=ising_hamiltonian()
H_mpo=hamiltonian_to_mpo(H,max_bond_dim=200,number_of_sites=N)
print(H)
#print(H_mpo)
#print(mpo_to_operator(H_mpo,number_of_sites=N))

[[-15.   1.   1.   0.   1.   0.   0.   0.   1.   0.   0.   0.   0.   0.
    0.   0.]
 [  1.  -5.   0.   1.   0.   1.   0.   0.   0.   1.   0.   0.   0.   0.
    0.   0.]
 [  1.   0.   5.   1.   0.   0.   1.   0.   0.   0.   1.   0.   0.   0.
    0.   0.]
 [  0.   1.   1.  -5.   0.   0.   0.   1.   0.   0.   0.   1.   0.   0.
    0.   0.]
 [  1.   0.   0.   0.   5.   1.   1.   0.   0.   0.   0.   0.   1.   0.
    0.   0.]
 [  0.   1.   0.   0.   1.  15.   0.   1.   0.   0.   0.   0.   0.   1.
    0.   0.]
 [  0.   0.   1.   0.   1.   0.   5.   1.   0.   0.   0.   0.   0.   0.
    1.   0.]
 [  0.   0.   0.   1.   0.   1.   1.  -5.   0.   0.   0.   0.   0.   0.
    0.   1.]
 [  1.   0.   0.   0.   0.   0.   0.   0.  -5.   1.   1.   0.   1.   0.
    0.   0.]
 [  0.   1.   0.   0.   0.   0.   0.   0.   1.   5.   0.   1.   0.   1.
    0.   0.]
 [  0.   0.   1.   0.   0.   0.   0.   0.   1.   0.  15.   1.   0.   0.
    1.   0.]
 [  0.   0.   0.   1.   0.   0.   0.   0.   0.   1.   1.   5.   0

In [47]:
H_mpo=hamiltonian_to_mpo(ising_hamiltonian(),max_bond_dim=200,number_of_sites=N)
mps=wavefunction_to_mps()
mps_dag=[A.conj() for A in mps]

def mps_norm(mps):
    env=np.array([[1.]])
    for A in mps:
        env=np.einsum(
            
            "ab,acd, bce->de",env,A.conj(),A
        )
    return env[0,0]

In [48]:
def left_right_environment(mpo,mps,number_of_sites=4):
    number_of_envs = number_of_sites // 2 + 1
    right_tensors = [None] * number_of_envs
    left_tensors = [None] * number_of_envs

    # Boundary environments
    left_tensors[0] = np.ones((1, 1, 1))
    right_tensors[0] = np.ones((1, 1, 1))
    for j, i in enumerate(range(0, number_of_sites - 1, 2)):
        L = left_tensors[j]

        # Contract site i
        L = np.einsum(
            "abc,asd,bste,ctf->def",
            L,
            mps[i].conj(),
            mpo[i],
            mps[i]
        )

        # Contract site i+1
        L = np.einsum(
            "abc,asd,bste,ctf->def",
            L,
            mps[i+1].conj(),
            mpo[i+1],
            mps[i+1]
        )

        left_tensors[j+1] = L

    for j, i in enumerate(range(number_of_sites - 2, -1, -2)):

        R = right_tensors[j]

        # Contract site i+1
        R = np.einsum(
            "asd,bste,ctf,def->abc",
            mps[i+1].conj(),
            mpo[i+1],
            mps[i+1],
            R
        )

        # Contract site i
        R = np.einsum(
            "asd,bste,ctf,def->abc",
            mps[i].conj(),
            mpo[i],
            mps[i],
            R
        )

        right_tensors[j+1] = R

    # Reverse right environments so that
    # right_tensors[j] corresponds to cut 2*j
    right_tensors.reverse()
    return left_tensors,right_tensors
l,r=left_right_environment(H_mpo,mps)

def update_left_environment(L, mpo_i, mpo_ip1, mps_i, mps_ip1):
    # Add site i
    L = np.einsum(
        "abc,asd,bste,ctf->def",
        L,
        mps_i.conj(),
        mpo_i,
        mps_i
    )

    # Add site i+1
    L = np.einsum(
        "abc,asd,bste,ctf->def",
        L,
        mps_ip1.conj(),
        mpo_ip1,
        mps_ip1
    )

    return L


def update_right_environment(R, mpo_i, mpo_ip1, mps_i, mps_ip1):
    # Add site i+1
    R = np.einsum(
        "asd,bste,ctf,def->abc",
        mps_ip1.conj(),
        mpo_ip1,
        mps_ip1,
        R
    )

    # Add site i
    R = np.einsum(
        "asd,bste,ctf,def->abc",
        mps_i.conj(),
        mpo_i,
        mps_i,
        R
    )

    return R

def dmrg(mpo,mps,number_of_sites=4,iterations=10):
    left_tensors,right_tensors=left_right_environment(mpo,mps,number_of_sites=number_of_sites)
    
    
    def matrix_vector_action(vector):
        theta=vector.reshape(left_bond_dim,2,2,right_bond_dim)
        result=np.einsum(
            "amb,msun,ntvr,grh,buvh->astg",
            L,W1,W2,R,theta
        )
        return result.reshape(-1)

    for l in range(iterations):
        for i in range(0, len(mps)-1, 2):
            theta=np.tensordot(mps[i],mps[i+1],([-1],[0]))
            theta_shape=theta.shape
            left_bond_dim=theta_shape[0]
            right_bond_dim=theta_shape[-1]
            W1=mpo[i]
            W2=mpo[i+1]
            j=i//2
            L=left_tensors[j]
            R=right_tensors[j+1]
            n=theta.size
            Heff = sp.sparse.linalg.LinearOperator(
            (n, n),
            matvec=matrix_vector_action,
            dtype=np.complex128 
            )

            eigenvalues, eigenvectors = sp.sparse.linalg.eigsh(
            Heff,
            k=1,
            which='SA',
            v0=theta.reshape(-1)
            )

            E0 = eigenvalues[0]

            theta_opt = eigenvectors[:, 0].reshape(
                left_bond_dim, 2, 2, right_bond_dim
            )

            U,S,Vh=np.linalg.svd(theta_opt.reshape(left_bond_dim*2,-1),full_matrices=False)
            mps[i]=U.reshape(left_bond_dim,2,len(S))
            M=np.diag(S)@Vh
            mps[i+1]=M.reshape(len(S),2,right_bond_dim)

            ####
            left_tensors[j+1] = update_left_environment(
                left_tensors[j],
                mpo[i],
                mpo[i+1],
                mps[i],
                mps[i+1]
            )
    return mps

In [51]:
mps_opt=dmrg(H_mpo,mps,number_of_sites=N,iterations=1000)
H=mpo_to_operator(H_mpo)
psi_opt=mps_to_state(mps_opt)
print((psi_opt.conj()@H@psi_opt)/(psi_opt.conj()@psi_opt))
energy = (
    np.vdot(psi_opt, H @ psi_opt)
    / np.vdot(psi_opt, psi_opt)
)
print(energy)

(-15.300829108049436+0j)
(-15.300829108049436+0j)


In [80]:
psi_opt = mps_to_state(mps_opt)
H = mpo_to_operator(H_mpo)

E1 = (
    psi_opt.conj() @ H @ psi_opt
    / (psi_opt.conj() @ psi_opt)
)

E2 = (
    np.vdot(psi_opt, H @ psi_opt)
    / np.vdot(psi_opt, psi_opt)
)

print("E1 =", E1)
print("E2 =", E2)
print("difference =", E1 - E2)

E1 = (-9.268631610630663+0j)
E2 = (-9.268631610630663+0j)
difference = 0j


# tdvp implemtation

In [18]:
def left_right_env2(mps,mpo,center_index):
    L=np.ones((1,1,1))
    R=np.ones((1,1,1))
    #left env loop
    for i in range(center_index):
        L=np.einsum(
            "abc,asd,bste,ctf->def",
            L,
            mps[i].conj(),
            mpo[i],
            mps[i]
        )
    for i in range(len(mps)-1, center_index, -1):
        n=len(mps)
        R = np.einsum(
            "asd,bste,ctf,def->abc",
            mps[i].conj(),
            mpo[i],
            mps[i],
            R
        )
    return L,R
        
def action_H_eff(mps,mpo,center_index):
    L,R=left_right_env2(mps,mpo,center_index)
    return np.einsum(
        "abc,csd,btse,ged->atg",
        L,mps[center_index],mpo[center_index],R
    )

def time_evolution(L,R,mps_tensor,mpo_tensor,deltat=0.01):
    H_eff = np.einsum(
    "abc,btse,ged->atgcsd",
    L, mpo_tensor, R
    )
    A=mps_tensor.reshape(-1)
    H_eff_matrix=H_eff.reshape(len(A),len(A))
    A=sp.linalg.expm(-1j*H_eff_matrix*deltat)@A
    return A.reshape(mps_tensor.shape[0],mps_tensor.shape[1],mps_tensor.shape[2])

    

In [ ]:
def tdvp_evolution(mps,mpo,time,dt=0.001):
    
    for n in range(math.floor(time/dt)):
        for i in range(len(mps)):
            tdvp_canonical_form(mps,i)
            L,R=left_right_env2(mps,mpo,i)
            mps[i]=time_evolution(L,R,mps[i],mpo[i],dt)
        for i in range(len(mps)-1, -1, -1):
            tdvp_canonical_form(mps,i)
            L,R=left_right_env2(mps,mpo,i)
            mps[i]=time_evolution(L,R,mps[i],mpo[i],dt)
    return mps

In [20]:
def compare_evolution(mps, H_mpo, H, psi0, times, dt):
    
    results = []

    for t in times:

        # TDVP
        mps_t = wavefunction_to_mps(psi0)
        mps_t = tdvp_evolution(mps_t, H_mpo, t, dt)
        psi_tdvp = mps_to_state(mps_t)

        # Exact
        psi_exact = sp.linalg.expm(-1j * H * t) @ psi0

        # Norms
        norm_tdvp = np.linalg.norm(psi_tdvp)
        norm_exact = np.linalg.norm(psi_exact)

        # Overlap
        overlap = np.vdot(psi_exact, psi_tdvp)

        # Fidelity
        fidelity = abs(overlap)**2

        results.append(
            [t, norm_tdvp, norm_exact, abs(overlap), fidelity]
        )

    return np.array(results)


In [28]:
times = np.linspace(0,3,4)
psi0=np.zeros(16)
psi0[0]=1
#psi0=wavefunction_to_mps(psi0)

results = compare_evolution(
    mps,
    H_mpo,
    H,
    psi0,
    times,
    dt=0.1
)

print(results)

[[0.         1.         1.         1.         1.        ]
 [1.         1.         1.         0.98096424 0.96229084]
 [2.         1.         1.         0.97527831 0.95116777]
 [3.         1.         1.         0.97466529 0.94997243]]
